# 🐝 Kinematyka – część 1: Gdzie jest pszczoła?

W tym zeszycie nauczysz się **opisywać położenie ciała** w przestrzeni – i przy okazji zrobisz pierwsze kroki w Pythonie.

**Jak z niego korzystać?** Komórki z kodem uruchamiasz przyciskiem ▶ po lewej stronie (albo `Shift + Enter`). Rób to po kolei, od góry do dołu. Nic się nie zepsuje – możesz śmiało zmieniać liczby i patrzeć, co się stanie.

> 💾 Chcesz zachować swoje zmiany? Wybierz *Plik → Zapisz kopię na Dysku*.

---
## 1. Układ odniesienia

Powiedzieć „pszczoła jest tutaj" to za mało. **Gdzie jest „tutaj"?** Żeby opisać położenie, musimy się umówić:

1. **Od czego mierzymy** – wybieramy punkt, który nazywamy *początkiem układu współrzędnych*.
2. **W jakich kierunkach mierzymy** – wybieramy trzy prostopadłe osie: $x$, $y$ i $z$.

Taki zestaw nazywamy **układem odniesienia** (tu: układem kartezjańskim). Położenie pszczoły to trzy liczby:

- $x$ – ile metrów od początku w kierunku osi $x$,
- $y$ – ile metrów w kierunku osi $y$,
- $z$ – ile metrów w górę, w kierunku osi $z$.

Liczby mogą być ujemne – wtedy jesteśmy „po drugiej stronie" początku układu.

## 2. Przygotowanie

Poniższa komórka przygotowuje rysowanie pszczoły. Na razie tylko ją uruchom – to kod pomocniczy, do którego wrócimy pod koniec zeszytu. 🙂

In [ ]:
#@title 🔧 Kod pomocniczy – rysowanie pszczoły (uruchom i nie przejmuj się nim, wrócimy do niego później)
import numpy as np
import plotly.graph_objects as go

R_PSZCZOLY = 0.35   # promień pszczoły [m]
L = 5               # osie rysujemy od -L do L
STRZALKA_R = "r\u20d7"   # litera r ze strzałką nad nią

def zbuduj_slady(x, y, z):
    """Zwraca listę wszystkich elementów rysunku dla pszczoły w punkcie (x, y, z)."""
    # --- pszczoła: kula w paski ---
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(0, np.pi, 40))
    pszczola = go.Surface(
        x=x + R_PSZCZOLY*np.cos(u)*np.sin(v),
        y=y + R_PSZCZOLY*np.sin(u)*np.sin(v),
        z=z + R_PSZCZOLY*np.cos(v),
        surfacecolor=(np.floor(v/(np.pi/7)) % 2),
        colorscale=[[0, "#FFD400"], [0.5, "#FFD400"], [0.5, "#1a1a1a"], [1, "#1a1a1a"]],
        cmin=0, cmax=1, showscale=False, hoverinfo="skip",
        lighting=dict(ambient=0.9, diffuse=0.3, specular=0.05),
    )

    # --- wektor położenia r: trzon + grot ---
    d = np.sqrt(x**2 + y**2 + z**2)               # długość wektora
    e = np.array([x, y, z]) / d if d > 1e-9 else np.array([0, 0, 1.0])
    dlugosc_grotu = min(0.6, max(d - R_PSZCZOLY, 0))   # grot kończy się na powierzchni pszczoły,
    promien_grotu = 0.2 * dlugosc_grotu / 0.6          # więc nie chowa się w jej wnętrzu
    ostrze = e * max(d - R_PSZCZOLY, 0)                # czubek grotu
    podstawa = ostrze - e * dlugosc_grotu              # środek podstawy grotu

    trzon = go.Scatter3d(x=[0, podstawa[0]], y=[0, podstawa[1]], z=[0, podstawa[2]],
                         mode="lines", line=dict(color="red", width=7), hoverinfo="skip")

    # grot = stożek zbudowany z trójkątów (apex + okrąg podstawy)
    a = np.cross(e, [1, 0, 0]) if abs(e[0]) < 0.9 else np.cross(e, [0, 1, 0])
    a = a / np.linalg.norm(a)
    b = np.cross(e, a)
    t = np.linspace(0, 2*np.pi, 16, endpoint=False)
    kolo = podstawa + promien_grotu*(np.outer(np.cos(t), a) + np.outer(np.sin(t), b))
    punkty = np.vstack([ostrze, kolo, podstawa])       # 0 = czubek, 1..16 = okrąg, 17 = środek
    n = len(t)
    boki = [(0, 1 + k, 1 + (k + 1) % n) for k in range(n)]
    spod = [(n + 1, 1 + (k + 1) % n, 1 + k) for k in range(n)]
    ij = np.array(boki + spod)
    grot = go.Mesh3d(x=punkty[:, 0], y=punkty[:, 1], z=punkty[:, 2],
                     i=ij[:, 0], j=ij[:, 1], k=ij[:, 2],
                     color="red", flatshading=True, hoverinfo="skip")

    etykieta = go.Scatter3d(x=[x/2], y=[y/2], z=[z/2 + 0.4], mode="text",
                            text=[STRZALKA_R], textfont=dict(color="red", size=24), hoverinfo="skip")

    # --- droga po osiach: 0 -> x -> y -> z (składowe wektora) ---
    skladowe = go.Scatter3d(x=[0, x, x, x], y=[0, 0, y, y], z=[0, 0, 0, z], mode="lines",
                            line=dict(color="steelblue", width=4, dash="dash"), hoverinfo="skip")

    # --- osie układu współrzędnych ---
    osie = go.Scatter3d(x=[-L, L, None, 0, 0, None, 0, 0], y=[0, 0, None, -L, L, None, 0, 0],
                        z=[0, 0, None, 0, 0, None, -L, L], mode="lines",
                        line=dict(color="gray", width=3), hoverinfo="skip")

    return [pszczola, trzon, grot, etykieta, skladowe, osie]

def tytul(x, y, z):
    return f"{STRZALKA_R} = ({x:g}; {y:g}; {z:g}) m"

# kamera: domyślny widok i widoki z różnych stron
WIDOKI = {
    "↺ Reset widoku": dict(eye=dict(x=1.6, y=-1.6, z=1.0), up=dict(x=0, y=0, z=1)),
    "Z góry":        dict(eye=dict(x=0.0, y=0.001, z=2.6), up=dict(x=0, y=1, z=0)),
    "Z przodu":      dict(eye=dict(x=0.0, y=-2.6, z=0.0), up=dict(x=0, y=0, z=1)),
    "Z boku":        dict(eye=dict(x=2.6, y=0.0, z=0.0), up=dict(x=0, y=0, z=1)),
}

def uklad(x, y, z):
    os_ = lambda nazwa: dict(title=nazwa, range=[-L, L], dtick=1)
    return go.Layout(
        title=dict(text=tytul(x, y, z), x=0.5),
        scene=dict(xaxis=os_("x [m]"), yaxis=os_("y [m]"), zaxis=os_("z [m]"),
                   aspectmode="cube", camera=WIDOKI["↺ Reset widoku"]),
        height=650, margin=dict(l=0, r=0, t=60, b=0), showlegend=False,
        updatemenus=[dict(
            type="buttons", direction="right", x=0.0, y=1.0, xanchor="left", yanchor="bottom",
            buttons=[dict(label=nazwa, method="relayout", args=[{"scene.camera": kamera}])
                     for nazwa, kamera in WIDOKI.items()],
        )],
    )

def rysuj_pszczole(x, y, z):
    go.Figure(data=zbuduj_slady(x, y, z), layout=uklad(x, y, z)).show()

def pszczola_z_suwakami():
    from ipywidgets import FloatSlider, VBox
    from IPython.display import display
    try:                                   # w Colabie wykres-widżet wymaga tego włączenia
        from google.colab import output
        output.enable_custom_widget_manager()
    except Exception:
        pass
    sx = FloatSlider(min=-5, max=5, step=0.5, value=3, description="x [m]")
    sy = FloatSlider(min=-5, max=5, step=0.5, value=2, description="y [m]")
    sz = FloatSlider(min=-5, max=5, step=0.5, value=1, description="z [m]")
    wykres = go.FigureWidget(data=zbuduj_slady(3, 2, 1), layout=uklad(3, 2, 1))

    def aktualizuj(zmiana=None):
        nowe = zbuduj_slady(sx.value, sy.value, sz.value)
        with wykres.batch_update():
            for stary, nowy in zip(wykres.data, nowe):
                dane = nowy.to_plotly_json()
                dane.pop("type", None)
                stary.update(dane)
            wykres.layout.title.text = tytul(sx.value, sy.value, sz.value)

    for s in (sx, sy, sz):
        s.observe(aktualizuj, names="value")
    display(VBox([sx, sy, sz, wykres]))

## 3. Pierwsza pszczoła

W Pythonie możemy nadać liczbie **nazwę**. Robimy to znakiem `=`. Poniżej mówimy: *pszczoła jest 3 m od początku w kierunku $x$, 2 m w kierunku $y$ i 1 m nad podłogą*.

Uruchom komórkę:

In [ ]:
x = 3
y = 2
z = 1

rysuj_pszczole(x, y, z)

### 🖱️ Jak obracać wykres?
- **Obrót:** przeciągnij wykres myszą (na telefonie – palcem).
- **Przybliżanie:** kółko myszy (na telefonie – dwa palce).
- **Przesuwanie:** przytrzymaj `Shift` i przeciągaj.
- **Powrót do początkowego widoku:** przycisk **↺ Reset widoku** nad wykresem (albo ikonka domku w pasku narzędzi w prawym górnym rogu).
- **Gotowe widoki:** *Z góry*, *Z przodu*, *Z boku*.

Na rysunku widzisz:
- **żółto-czarną kulę** – pszczołę (traktujemy ją jak punkt, bo jest bardzo mała w porównaniu z odległościami),
- **niebieską przerywaną linię** – „drogę po osiach": najpierw $x$ metrów wzdłuż osi $x$, potem $y$ metrów równolegle do osi $y$, na koniec $z$ metrów w górę,
- **czerwoną strzałkę** – to jest **wektor położenia** $\vec{r}$.

> ✏️ **Zadanie 1.** Zmień liczby `x`, `y`, `z` (np. na `-2`, `4`, `3`) i uruchom komórkę jeszcze raz. Co się dzieje, gdy któraś z liczb jest ujemna? A gdy wszystkie są równe `0`?

> ✏️ **Zadanie 2.** Kliknij przycisk **Z góry**. Którą współrzędną pszczoły *przestałeś widzieć*? A co zobaczysz po kliknięciu **Z przodu**? Wróć do widoku początkowego przyciskiem **↺ Reset widoku**.

## 4. Wektor położenia $\vec{r}$

Trzy liczby $x$, $y$, $z$ opisują położenie pszczoły. Zbierzmy je w jeden obiekt – **wektor położenia**:

$$\vec{r} = (x;\ y;\ z)$$

Wektor oznaczamy **strzałką nad literą**: $\vec{r}$. To podpowiedź, że ma on nie tylko *wartość*, ale i *kierunek oraz zwrot*:

- **początek** wektora $\vec{r}$ leży w początku układu współrzędnych,
- **koniec** wektora (grot strzałki) leży w miejscu, w którym jest pszczoła,
- liczby $x$, $y$, $z$ nazywamy **składowymi** wektora.

W Pythonie wektor zapisujemy jako listę liczb w nawiasie kwadratowym, zapakowaną w `np.array(...)`:

In [ ]:
r = np.array([x, y, z])

print("Wektor położenia r =", r)
print("Składowa x:", r[0])
print("Składowa y:", r[1])
print("Składowa z:", r[2])

> 💡 **Uwaga o Pythonie:** numeracja elementów zaczyna się od **0**, nie od 1. Dlatego `r[0]` to $x$, `r[1]` to $y$, a `r[2]` to $z$.

### Jak daleko jest pszczoła od początku układu?

Długość wektora $\vec{r}$ to odległość pszczoły od początku układu. Liczymy ją z twierdzenia Pitagorasa w przestrzeni:

$$r = |\vec{r}| = \sqrt{x^2 + y^2 + z^2}$$

Zwróć uwagę: **bez strzałki** ($r$) to zwykła liczba – długość. **Ze strzałką** ($\vec{r}$) to wektor – ma kierunek i zwrot.

In [ ]:
dlugosc = np.sqrt(x**2 + y**2 + z**2)

print(f"Pszczoła jest {dlugosc:.2f} m od początku układu")

> 💡 **Uwaga o Pythonie:** `**2` oznacza „do kwadratu", `np.sqrt` to pierwiastek kwadratowy. W tekście w cudzysłowie `{...}` wstawia wartość zmiennej, a `:.2f` zaokrągla ją do dwóch miejsc po przecinku.

> ✏️ **Zadanie 3.** Policz **w głowie** długość wektora dla $\vec{r} = (3;\ 4;\ 0)$ m. Potem wpisz `x = 3`, `y = 4`, `z = 0` w komórkach wyżej i sprawdź, czy Python się z Tobą zgadza. Znasz tę trójkę liczb skądś? 😉

## 5. Bawimy się suwakami

Zamiast co chwilę zmieniać liczby w kodzie, użyjmy suwaków. Przesuwaj je i obserwuj, jak zmienia się pszczoła i wektor $\vec{r}$. Wykres możesz przy tym obracać – widok **nie zresetuje się** po przesunięciu suwaka.

In [ ]:
pszczola_z_suwakami()

> ⚠️ Jeśli suwaki się pojawiły, ale wykres nie reaguje – uruchom komórkę ponownie (czasem Colab potrzebuje chwili na przygotowanie widżetów).

> ✏️ **Zadanie 4 (eksperymenty).** Ustaw suwaki tak, aby:
> 1. pszczoła leżała **na osi $x$** (które składowe muszą być zerowe?),
> 2. pszczoła leżała **w płaszczyźnie $xy$** (czyli „na podłodze"),
> 3. wektor $\vec{r}$ był skierowany **dokładnie w dół** od początku układu,
> 4. pszczoła była **w tej samej odległości** od początku, co na początku, ale w zupełnie innym miejscu. Ile jest takich miejsc?

> 🤔 **Pytanie do przemyślenia.** Czy położenie pszczoły zależy od tego, jak ustawimy układ współrzędnych? Co się stanie z liczbami $x$, $y$, $z$, jeśli przesuniemy początek układu o 1 m w bok? A z samą pszczołą?

## 6. Podsumowanie

| Pojęcie | Znaczenie | W Pythonie |
|---|---|---|
| Układ odniesienia | początek + trzy osie $x$, $y$, $z$ | – |
| Współrzędne | trzy liczby opisujące położenie | `x = 3` |
| Wektor położenia $\vec{r}$ | strzałka od początku układu do ciała | `r = np.array([x, y, z])` |
| Długość wektora $r = \lvert\vec{r}\rvert$ | odległość ciała od początku układu | `np.sqrt(x**2 + y**2 + z**2)` |

**W następnej części:** pszczoła zaczyna *lecieć*. Zobaczymy, jak zmienia się wektor $\vec{r}$ w czasie, i zdefiniujemy **przemieszczenie** oraz **prędkość**.

---
### 🔎 Dla ciekawskich: co robi kod pomocniczy?
Rozwiń komórkę „Kod pomocniczy" (dwukrotne kliknięcie w jej tytuł albo ikonka `<>`). Rysunek powstaje z biblioteki `plotly`: kula w paski to `go.Surface`, linie to `go.Scatter3d`, a grot strzałki to stożek zbudowany z trójkątów (`go.Mesh3d`). Spróbuj zmienić kolory pszczoły albo jej rozmiar `R_PSZCZOLY`.